In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (r2_score, mean_absolute_error, root_mean_squared_error)

In [5]:
df=pd.read_csv('cleaned_MH_OC.csv')

In [6]:
embed_df = pd.read_csv("alphaearth_embeddings.csv")
df_embed = df.merge(embed_df,on='fid',how='left')
df_embed.head()

,lat_key,fid,latitude,longitude,CLIMATE_VALUE,CLIMATE_SUBCLASS,CLIMATE_CLASS,DOMSOI,SOIL_TYPE,SOIL_SUBCLASS,...,A55,A56,A57,A58,A59,A60,A61,A62,A63,embedding_year
0,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,-0.055363,-0.010396,-0.059116,-0.179377,-0.186082,0.079723,-0.062991,0.038447,0.088827,2017
1,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,-0.066990,-0.029773,-0.062991,-0.141730,-0.166336,0.119093,-0.066990,0.032541,0.079723,2018
2,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,-0.032541,-0.035433,-0.044844,-0.147697,-0.186082,0.113741,-0.041584,0.019931,0.048228,2019
3,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,-0.066990,-0.032541,-0.062991,-0.147697,-0.130165,0.093564,-0.062991,0.027128,0.093564,2020
4,15.709677,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,-0.055363,-0.022207,-0.108512,-0.141730,-0.084214,0.066990,-0.084214,0.041584,0.088827,2021


Separate Alphaembedding columns

In [7]:
embedding_cols = [
    col
    for col in df_embed.columns
    if col.startswith("A")
]

print("Number of embeddings:", len(embedding_cols))
print(embedding_cols[:10])

Number of embeddings: 64
['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']


In [8]:
soil_counts = (
    df_embed["SOIL_TYPE"]
    .value_counts()
    .reset_index()
)

soil_counts.columns = ["SOIL_TYPE", "Samples"]

soil_counts

,SOIL_TYPE,Samples
0,VERTISOLS,176742
1,CAMBISOLS,34533
2,NITOSOLS,29088
3,PHAEOZEMS,15903
4,LITHOSOLS,10476
5,LUVISOLS,9234
6,ACRISOLS,3456
7,FLUVISOLS,1800
8,PLANOSOLS,18


In [9]:
df_embed.groupby("fid").size().value_counts()

9     28685
36      603
81       17
Name: count, dtype: int64

In [10]:
embedding_cols = [c for c in df_embed.columns if c.startswith("A")]

embed_wide = (
    df_embed[
        ["fid", "embedding_year"] + embedding_cols
    ]
    .pivot_table(
        index="fid",
        columns="embedding_year",
        values=embedding_cols
    )
)

embed_wide.columns = [
    f"{feature}_{year}"
    for feature, year in embed_wide.columns
]

embed_wide = embed_wide.reset_index()

In [11]:
soc_info = (
    df_embed
    .drop_duplicates("fid")
    [["fid", "OC", "SOIL_TYPE"]]
)

In [12]:
df_model = soc_info.merge(
    embed_wide,
    on="fid",
    how="inner"
)

print(df_model.shape)

(29305, 579)


In [15]:
df_model.head()

,fid,OC,SOIL_TYPE,A00_2017,A00_2018,A00_2019,A00_2020,A00_2021,A00_2022,A00_2023,...,A62_2025,A63_2017,A63_2018,A63_2019,A63_2020,A63_2021,A63_2022,A63_2023,A63_2024,A63_2025
0,48980.0,1.590,ACRISOLS,-0.038447,-0.038447,-0.024606,-0.032541,-0.029773,-0.032541,-0.038447,...,0.071111,0.088827,0.079723,0.048228,0.093564,0.088827,0.098424,0.084214,0.051734,0.075356
1,48981.0,1.384,ACRISOLS,0.010396,0.006151,0.002215,0.015748,0.013841,0.003937,-0.006151,...,0.029773,0.059116,0.051734,0.006151,0.055363,0.059116,0.071111,0.048228,0.035433,0.062991
2,48982.0,1.316,ACRISOLS,-0.071111,-0.066990,-0.071111,-0.075356,-0.066990,-0.088827,-0.088827,...,0.141730,0.006151,-0.008858,-0.071111,-0.002215,-0.013841,0.017778,-0.012057,-0.038447,-0.032541
3,48839.5,0.869,NITOSOLS,-0.098424,-0.113741,-0.075356,-0.103406,-0.093564,-0.113741,-0.108512,...,0.113741,0.075356,0.062991,-0.003937,0.051734,0.022207,0.041584,0.071111,0.022207,0.079723
4,48835.0,0.824,NITOSOLS,-0.098424,-0.113741,-0.075356,-0.103406,-0.093564,-0.113741,-0.108512,...,0.113741,0.075356,0.062991,-0.003937,0.051734,0.022207,0.041584,0.071111,0.022207,0.079723


In [13]:
feature_cols = [
    c
    for c in df_model.columns
    if c.startswith("A")
]

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

results = []

for soil in sorted(df_model["SOIL_TYPE"].dropna().unique()):

    soil_df = df_model[df_model["SOIL_TYPE"] == soil]

    if len(soil_df) < 100:
        continue

    X = soil_df[feature_cols]
    y = soil_df["OC"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    results.append({
        "SOIL_TYPE": soil,
        "Samples": len(soil_df),
        "R2": r2_score(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values(
    "R2",
    ascending=False
)

results_df

,SOIL_TYPE,Samples,R2,RMSE,MAE
5,NITOSOLS,3062,0.675803,0.450511,0.328376
2,FLUVISOLS,190,0.461617,0.485105,0.400605
1,CAMBISOLS,3586,0.376152,0.304804,0.186779
3,LITHOSOLS,1098,0.276101,0.225172,0.155478
6,PHAEOZEMS,1627,0.213049,0.282748,0.177903
4,LUVISOLS,970,0.184939,0.255719,0.154142
7,VERTISOLS,18408,0.184119,0.238190,0.153039
0,ACRISOLS,362,0.068408,0.268117,0.210595
